<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L4/01_finetune_frozen_base_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Fine-tuning by Replacing the FC Layer (Frozen Base)

**Goal:** classify 5 types of flowers using a ResNet18 that was pre-trained on ImageNet, training **only a new final layer**.

### Main ideas in this notebook
1. **Transfer learning** — reuse a network that already learned general visual features from 1.2 million ImageNet images.
2. **Freeze the convolutional base** — its weights are not updated (`requires_grad = False`).
3. **Replace the fully-connected (FC) layer** — ImageNet's 1000-class head is swapped for a new 5-class head.
4. **Speed-up trick: extract features once** — since the base is frozen, its output never changes. We can compute the features **once**, cache them, and then train *many* small classifiers in seconds.

> 💡 A GPU is recommended (e.g. Google Colab → *Runtime → Change runtime type → GPU*), but everything also runs on CPU, just slower.

## 1. Setup

In [ ]:
import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.datasets.utils import download_and_extract_archive

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

# Pick the fastest available device: NVIDIA GPU -> Apple Silicon GPU -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("PyTorch", torch.__version__, "| device:", device)

ModuleNotFoundError: No module named 'matplotlib'

## 2. The dataset — TensorFlow *flower_photos*

About 3,670 photos in 5 classes (daisy, dandelion, roses, sunflowers, tulips). Each class is a folder, which is exactly the layout `torchvision.datasets.ImageFolder` expects.

In [ ]:
DATA_URL  = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_ROOT = "data"
DATA_DIR  = os.path.join(DATA_ROOT, "flower_photos")

if not os.path.isdir(DATA_DIR):
    download_and_extract_archive(DATA_URL, download_root=DATA_ROOT)   # ~220 MB, only downloaded once

# One sub-folder per class -> ImageFolder labels the images automatically
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
for c in class_names:
    print(f"{c:12s} {len(os.listdir(os.path.join(DATA_DIR, c)))} images")

### Pre-processing

A pre-trained network expects its inputs to look like the data it was trained on. For ImageNet models this means:

* resize/crop to **224 × 224**
* convert to a tensor in [0, 1]
* **normalise** each colour channel with the ImageNet mean and std

In this notebook we use the same deterministic transform for training and validation (no data augmentation — that comes in Notebook 3). This matters for the feature-caching trick later.

In [ ]:
# ResNet18 was trained on ImageNet, so we must feed it images prepared the SAME way:
# 224x224 pixels, normalised with the ImageNet channel mean and standard deviation.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
def make_datasets(train_tf, val_tf, val_frac=0.2, seed=SEED):
    """Same fixed 80/20 split in every notebook, so results are comparable."""
    n = len(datasets.ImageFolder(DATA_DIR))
    idx = np.random.RandomState(seed).permutation(n)
    n_val = int(val_frac * n)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    # Two ImageFolder objects over the same files so train/val can use different transforms
    train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)
    return train_ds, val_ds

BATCH_SIZE  = 32
NUM_WORKERS = 2          # set to 0 if you get DataLoader errors (e.g. on Windows)
PIN = device.type == "cuda"

def make_loaders(train_ds, val_ds, shuffle_train=True):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=shuffle_train,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    return train_loader, val_loader

train_ds, val_ds = make_datasets(train_tf=eval_tf, val_tf=eval_tf)
train_loader, val_loader = make_loaders(train_ds, val_ds)
print(f"train: {len(train_ds)} images | val: {len(val_ds)} images | classes: {class_names}")

Let's look at a few training images (we undo the normalisation just for display).

In [ ]:
def show_batch(ds, n=8, title=None):
    mean, std = torch.tensor(IMAGENET_MEAN).view(3,1,1), torch.tensor(IMAGENET_STD).view(3,1,1)
    idx = np.random.choice(len(ds), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2.2*n, 2.6))
    for ax, i in zip(axes, idx):
        img, label = ds[i]
        ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0))
        ax.set_title(class_names[label]); ax.axis("off")
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()

show_batch(train_ds)

## 3. Inside ResNet18

ResNet18 has two parts:

| Part | Layers | Role |
|---|---|---|
| **Convolutional base** (feature extractor) | `conv1 → bn1 → relu → maxpool → layer1 → layer2 → layer3 → layer4 → avgpool` | turns a 224×224 image into a **512-number feature vector** |
| **Classifier head** | `fc` = `Linear(512 → 1000)` | maps the 512 features to the 1000 ImageNet classes |

Early layers detect generic patterns (edges, colours, textures); later layers detect more complex, task-specific shapes. Because those features are useful for *almost any* image task, we can keep the base and just learn a new head.

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

# Print only the top-level blocks to keep it readable
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"{name:8s} {module.__class__.__name__:18s} params: {n_params:>10,}")
print("\nOriginal head:", model.fc)

### The training loop

A few helper functions used in all three notebooks:

* `run_epoch` — one pass over the data (train or evaluate)
* `fit` — trains for several epochs, records the history and keeps the best weights
* `set_train_mode` — like `model.train()`, **but keeps frozen BatchNorm layers in eval mode**. Otherwise BatchNorm would keep updating its running statistics and the "frozen" base would actually drift.

We only pass the **trainable** parameters (`model.fc.parameters()`) to the optimizer.

In [ ]:
def set_train_mode(model):
    """model.train(), but keep BatchNorm layers of FROZEN blocks in eval mode.

    In train mode BatchNorm keeps updating its running mean/variance even when its
    weights are frozen - that would silently change the 'frozen' features.
    """
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. Trains if an optimizer is given, otherwise just evaluates."""
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += x.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, optimizer, epochs, scheduler=None, verbose=True):
    """Standard training loop. Returns the history and restores the best weights (by val acc)."""
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
    best_acc, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        set_train_mode(model)
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        model.eval()
        with torch.no_grad():
            vl, va = run_epoch(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        dt = time.time() - t0
        for k, v in zip(hist, (tl, ta, vl, va, dt)):
            hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {ep:2d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | "
                  f"val loss {vl:.3f} acc {va:.3f} | {dt:.1f}s")
    model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    return hist

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
    return trainable

## 4. Approach A — Freeze the base and replace the FC layer

Two steps:

1. **Freeze** every existing parameter: `param.requires_grad = False`. Autograd will not compute gradients for them, and the optimizer will not change them.
2. **Replace** `model.fc` with a new `nn.Linear(512, 5)`. A newly created layer has `requires_grad=True` by default, so it is the *only* trainable part.

Notice how few parameters we actually train.

In [ ]:
def build_frozen_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():          # 1) freeze the whole pre-trained network
        p.requires_grad = False
    in_features = model.fc.in_features    # 512 for ResNet18
    model.fc = nn.Linear(in_features, num_classes)   # 2) new head (trainable)
    return model.to(device)

model_A = build_frozen_resnet18(len(class_names))
print(model_A.fc)
count_params(model_A);

In [ ]:
EPOCHS_A = 5
set_seed()
model_A = build_frozen_resnet18(len(class_names))
optimizer = optim.Adam(model_A.fc.parameters(), lr=1e-3)

t0 = time.time()
hist_A = fit(model_A, train_loader, val_loader, optimizer, epochs=EPOCHS_A)
time_A = time.time() - t0
print(f"\nApproach A: best val acc = {hist_A['best_val_acc']:.3f}, total time = {time_A:.1f}s")

### Where does the time go?

Even though only ~2.5k parameters are trained, **every epoch still pushes every image through the entire ResNet18 base** (≈11 million parameters, ~1.8 GFLOPs per image). The base is frozen, so for a given image it produces **exactly the same 512 features every epoch** — we are recomputing the same thing over and over.

## 5. Approach B — Extract features once, then train many heads

Since the base is frozen *and* the input transforms are deterministic:

1. Remove the head (`model.fc = nn.Identity()`), so the network outputs the 512-d feature vector.
2. Run every image through the base **once** and store the features in memory (or on disk).
3. Train small classifiers on the cached `(features, label)` pairs. Each epoch is now just a tiny matrix multiplication → **seconds instead of minutes**.

This makes it cheap to try **many heads / hyper-parameters** (learning rate, weight decay, an extra hidden layer, …).

> ⚠️ This only works when the base is frozen and the inputs do not change between epochs. With random data augmentation (Notebook 3) or unfrozen conv layers (Notebook 2) the features change every epoch, so they cannot be cached.

In [ ]:
feature_extractor = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
feature_extractor.fc = nn.Identity()      # output = 512-d feature vector instead of class scores
feature_extractor = feature_extractor.to(device).eval()

@torch.no_grad()
def extract_features(model, loader):
    feats, labels = [], []
    for x, y in loader:
        feats.append(model(x.to(device)).cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

# shuffle=False is fine here: we shuffle the cached features later
train_loader_noshuf, _ = make_loaders(train_ds, val_ds, shuffle_train=False)

t0 = time.time()
X_train, y_train = extract_features(feature_extractor, train_loader_noshuf)
X_val,   y_val   = extract_features(feature_extractor, val_loader)
time_extract = time.time() - t0

print("train features:", tuple(X_train.shape), "| val features:", tuple(X_val.shape))
print(f"feature extraction took {time_extract:.1f}s (done ONCE)")

# Optional: save to disk so you never need to recompute them
torch.save({"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val,
            "classes": class_names}, "flower_resnet18_features.pt")

### Training a head on cached features

The cached features are small (`N × 512` floats), so we can put them in a `TensorDataset` and use a large batch size. The training loop is the same `fit` function — the "model" is now just the head.

In [ ]:
from torch.utils.data import TensorDataset

feat_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
feat_val_loader   = DataLoader(TensorDataset(X_val, y_val),     batch_size=512)

def train_head(head, lr=1e-3, weight_decay=0.0, epochs=30, verbose=False):
    set_seed()
    head = head.to(device)
    opt = optim.Adam(head.parameters(), lr=lr, weight_decay=weight_decay)
    t0 = time.time()
    hist = fit(head, feat_train_loader, feat_val_loader, opt, epochs=epochs, verbose=verbose)
    hist["time"] = time.time() - t0
    return head, hist

# Same setup as Approach A: a single linear layer, Adam lr=1e-3, same number of epochs
set_seed()
head, hist_B = train_head(nn.Linear(512, len(class_names)), lr=1e-3, epochs=EPOCHS_A, verbose=True)
print(f"\nApproach B: best val acc = {hist_B['best_val_acc']:.3f}, "
      f"head training time = {hist_B['time']:.2f}s (+ {time_extract:.1f}s one-off extraction)")

### Train *multiple* heads in seconds

Now the real benefit: we can run a small hyper-parameter search. Each experiment below trains for 30 epochs — something that would take many minutes per experiment with Approach A.

We try several **linear** heads (different learning rates / weight decay) and, for comparison, a small **MLP** head (one hidden layer + dropout).

In [ ]:
C = len(class_names)
experiments = {
    "linear  lr=1e-2":           (lambda: nn.Linear(512, C),                     dict(lr=1e-2)),
    "linear  lr=1e-3":           (lambda: nn.Linear(512, C),                     dict(lr=1e-3)),
    "linear  lr=1e-4":           (lambda: nn.Linear(512, C),                     dict(lr=1e-4)),
    "linear  lr=1e-3 wd=1e-3":   (lambda: nn.Linear(512, C),                     dict(lr=1e-3, weight_decay=1e-3)),
    "MLP 512-256-5 dropout 0.5": (lambda: nn.Sequential(nn.Linear(512, 256), nn.ReLU(),
                                                        nn.Dropout(0.5), nn.Linear(256, C)),
                                  dict(lr=1e-3)),
}

results = {}
for name, (make_head, kwargs) in experiments.items():
    set_seed()
    h, hist = train_head(make_head(), epochs=30, **kwargs)
    results[name] = (h, hist)
    print(f"{name:28s} best val acc {hist['best_val_acc']:.3f}  ({hist['time']:.2f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, (_, hist) in results.items():
    axes[0].plot(hist["train_loss"], label=name)
    axes[1].plot(hist["val_acc"], label=name)
axes[0].set_title("Training loss"); axes[1].set_title("Validation accuracy")
for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=.3)
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

### Put the best head back on top of ResNet18

A head trained on cached features is **fully compatible** with the original network: just attach it as `model.fc`. The result is a normal end-to-end image classifier, and its accuracy on raw images matches the accuracy measured on the cached features.

In [ ]:
best_name = max(results, key=lambda k: results[k][1]["best_val_acc"])
best_head = results[best_name][0]
print("best head:", best_name)

final_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
final_model.fc = best_head                 # plug the trained head in
final_model = final_model.to(device).eval()

with torch.no_grad():
    _, acc = run_epoch(final_model, val_loader, nn.CrossEntropyLoss())
print(f"end-to-end val accuracy on raw images: {acc:.3f}  "
      f"(on cached features: {results[best_name][1]['best_val_acc']:.3f})")

## 6. Speed comparison

In [ ]:
n_exp = len(experiments)
avg_head_time = np.mean([h["time"] for _, h in results.values()])
avg_A_epoch   = np.mean(hist_A["epoch_time"])

print(f"Approach A  (full forward pass every epoch): {avg_A_epoch:7.2f} s / epoch")
print(f"Approach B  (cached features)              : {avg_head_time/30:7.3f} s / epoch "
      f"(+ {time_extract:.1f}s one-off)")
print()
print(f"Cost of {n_exp} experiments x 30 epochs:")
print(f"  Approach A (estimated): {n_exp*30*avg_A_epoch/60:7.1f} min")
print(f"  Approach B (measured) : {(time_extract + sum(h['time'] for _, h in results.values()))/60:7.1f} min")

## 7. Summary

* **Transfer learning**: a pre-trained ResNet18 already provides strong, general image features; with a frozen base and a new 5-class FC layer we reach high accuracy after only a few epochs, training < 0.1 % of the parameters.
* **Freezing**: `requires_grad=False` on the base, and only the new `fc` parameters go to the optimizer. Keep frozen BatchNorm layers in `eval()` mode.
* **Feature extraction speed-up**: when the base is frozen and the inputs are deterministic, the base's output never changes → compute it **once**, then train as many heads as you like in seconds. Approach A and Approach B learn the *same* kind of model; B is just far cheaper.
* **Limitation**: the conv features were learned for ImageNet, not for flowers. In **Notebook 2** we let the last conv block(s) adapt to our data and compare.